# OCR Pipeline for Historical Print Periodicals

Automated full-text OCR for scanned periodicals using the Mistral OCR API (`mistral-ocr-latest`).

**Features:** Structured extraction (headings, paragraphs, footnotes) | Auto-splitting of large PDFs (>50 MB) | SQLite checkpoint system | Output as Markdown, Plain Text, and JSON

**Workflow:** Installation → Setup → Discovery → Batch OCR → Cleanup

**Documentation:** `docs/ARCHITECTURE.md` (System architecture) | `docs/LLM_WORKFLOW.md` (API workflow)

---

## Prerequisites

**Mistral API Key required:** Registration at [console.mistral.ai](https://console.mistral.ai) → API Keys → Create new key

**Configuration:** Add your API key to the `.env` file (project root):
```
MISTRAL_API_KEY=your_mistral_api_key_here
```

**Installation:** The following cell installs all Python packages automatically

---

## Cell 1: Setup and Imports

Loads Mistral AI SDK and local helper functions, configures logging, creates project directories (`data/input`, `data/output`, `data/tracking`), and initializes the SQLite database for the checkpoint system.

**Config via `.env`:** `MISTRAL_MODEL` (default: mistral-ocr-latest) | `DELAY_SECONDS` (rate limiting) | `MAX_RETRIES` | `TIMEOUT_SECONDS`

**Output:** Status messages with API key check, path overview, and configuration

In [ ]:
# Install dependencies (run once)
!pip install -r ../requirements.txt

# Imports
import os
import json
import logging
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from mistralai import Mistral

# Import local utils (explicit imports for better code readability)
from utils import (
    init_tracking_database,
    get_processed_pdfs,
    prepare_file_for_ocr,
    process_single_pdf,
    get_processing_stats,
    get_storage_stats,
    cleanup_temp_files
)

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Load API key
load_dotenv(Path("..") / ".env")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

if not MISTRAL_API_KEY:
    raise ValueError(
        "MISTRAL_API_KEY not found! "
        "Please create .env file in project root with: MISTRAL_API_KEY=your_key_here"
    )

# Initialize Mistral client
try:
    client = Mistral(api_key=MISTRAL_API_KEY)
    MISTRAL_MODEL = os.getenv("MISTRAL_MODEL", "mistral-ocr-latest")
except Exception as e:
    raise RuntimeError(
        f"Mistral client could not be initialized: {e}\n\n"
        "Possible causes:\n"
        "  - Mistral SDK not installed: pip install mistralai\n"
        "  - Invalid API key format\n"
        "  - Import error in mistralai package"
    ) from e

# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_INPUT = PROJECT_ROOT / "data" / "input"
DATA_OUTPUT = PROJECT_ROOT / "data" / "output"
DATA_TRACKING = PROJECT_ROOT / "data" / "tracking"

# Create directories if they don't exist
DATA_INPUT.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT.mkdir(parents=True, exist_ok=True)
DATA_TRACKING.mkdir(parents=True, exist_ok=True)

# Configuration
DELAY_SECONDS = float(os.getenv("DELAY_SECONDS", "2.0"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "5"))
TIMEOUT = int(os.getenv("TIMEOUT_SECONDS", "120"))

# Tracking database
DB_PATH = str(DATA_TRACKING / "ocr_progress.db")
init_tracking_database(DB_PATH)

# Status output
print("="*70)
print("OCR PIPELINE FOR HISTORICAL PERIODICALS")
print("="*70)
print(f"✓ Mistral client initialized")
print(f"✓ Model: {MISTRAL_MODEL}")
print(f"✓ API Key: {MISTRAL_API_KEY[:8]}...{MISTRAL_API_KEY[-4:]}")
print(f"✓ Tracking DB: {DB_PATH}")
print(f"")
print(f"📁 Paths:")
print(f"  Input:    {DATA_INPUT}")
print(f"  Output:   {DATA_OUTPUT}")
print(f"  Tracking: {DATA_TRACKING}")
print(f"")
print(f"⚙️ Configuration:")
print(f"  Delay:      {DELAY_SECONDS}s")
print(f"  Max Retry:  {MAX_RETRIES}")
print(f"  Timeout:    {TIMEOUT}s")
print("="*70)
print(f"✓ Setup complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

## Cell 2: File Discovery

Searches the input directory (`data/input/`) recursively for PDFs (`.pdf`) and images (`.jpg`, `.jpeg`, `.png`). Collects metadata (file path, type, size) and creates a list of all files to process.

**Output:** Statistics overview (number of PDFs, images, total size) and detailed list of all found files

**Note:** Large PDFs (>50 MB or >1000 pages) will be automatically split into 500-page chunks in Cell 3

In [ ]:
# Supported file formats
SUPPORTED_EXTENSIONS = {
    'pdf': ['.pdf'],
    'image': ['.jpg', '.jpeg', '.png']
}

# Find all files in input folder
all_files = []
skipped_files = []

for root, dirs, files in os.walk(DATA_INPUT):
    for filename in files:
        file_path = Path(root) / filename
        ext = file_path.suffix.lower()
        
        # Check if supported format
        file_type = None
        if ext in SUPPORTED_EXTENSIONS['pdf']:
            file_type = 'PDF'
        elif ext in SUPPORTED_EXTENSIONS['image']:
            file_type = 'Image'
        
        if file_type:
            try:
                # Get file size (can fail for symlinks, permission errors, etc.)
                size_mb = file_path.stat().st_size / (1024 * 1024)
                all_files.append({
                    'path': str(file_path),
                    'filename': filename,
                    'type': file_type,
                    'size_mb': round(size_mb, 2),
                    'extension': ext
                })
            except (OSError, PermissionError) as e:
                # File not accessible (symlink, permission, cloud sync, etc.)
                logger.warning(f"File skipped (no access): {filename} - {e}")
                skipped_files.append({'filename': filename, 'error': str(e)})
                continue

# Statistics
pdfs = [f for f in all_files if f['type'] == 'PDF']
images = [f for f in all_files if f['type'] == 'Image']
total_size = sum(f['size_mb'] for f in all_files)

# Output
print("="*70)
print("FILE DISCOVERY")
print("="*70)
print(f"📁 Input directory: {DATA_INPUT}")
print(f"")
print(f"📊 Statistics:")
print(f"  PDFs:   {len(pdfs)}")
print(f"  Images: {len(images)}")
print(f"  Total:  {len(all_files)} files")
print(f"  Size:   {total_size:.2f} MB")

if skipped_files:
    print(f"  ⚠️ Skipped: {len(skipped_files)} files (no access)")

print(f"")

if not all_files:
    print("⚠️  NO FILES FOUND!")
    print(f"   Please place PDFs or images in {DATA_INPUT}")
else:
    print(f"📄 Found files:")
    print(f"")
    for i, file_info in enumerate(all_files, 1):
        print(f"{i:3d}. [{file_info['type']:5s}] {file_info['filename']:50s} ({file_info['size_mb']:6.2f} MB)")

if skipped_files:
    print(f"")
    print(f"⚠️  Skipped files:")
    for skip in skipped_files[:5]:  # Show max 5
        print(f"  - {skip['filename']}: {skip['error']}")
    if len(skipped_files) > 5:
        print(f"  ... and {len(skipped_files) - 5} more")

print("="*70)

# Save list for further processing
FILES_TO_PROCESS = all_files

## Cell 3: Batch OCR Processing

Processes all files found in Cell 2 with the Mistral OCR API. The checkpoint system automatically skips already-processed files (resumable after interruption). Large PDFs are split into chunks (→ `data/tracking/pdf_chunks/`), OCR processing includes retry logic for API errors.

**Output:** Live progress display per file | Chunk statistics | Final overall statistics (files, pages, API calls, costs)

**Results:** `.md` (Markdown with structure) | `.txt` (Plain text) | `_metadata.json` (Confidence, page count, warnings) + SQLite DB update

**Resume:** If interrupted, re-run this cell → Checkpoint system skips completed files

In [ ]:
# Temp directory for split PDFs
TEMP_DIR = DATA_TRACKING / "pdf_chunks"
TEMP_DIR.mkdir(exist_ok=True)

# Load already processed files
processed_files = get_processed_pdfs(DB_PATH)

# Statistics
total_files = len(FILES_TO_PROCESS)
total_processed = 0
total_errors = 0
total_api_calls = 0
start_time = datetime.now()

print("="*70)
print("BATCH OCR PROCESSING")
print("="*70)
print(f"📊 Status:")
print(f"  Total files:        {total_files}")
print(f"  Already processed:  {len(processed_files)}")
print(f"  To do:              {total_files - len(processed_files)}")
print(f"")
print(f"🚀 Starting processing...")
print("="*70)
print(f"")

# Loop over all files
for file_idx, file_info in enumerate(FILES_TO_PROCESS, 1):
    file_path = file_info['path']
    file_id = Path(file_path).stem

    # Skip if already processed
    if file_id in processed_files:
        print(f"[{file_idx}/{total_files}] ⏭️  SKIP: {file_info['filename']} (already processed)")
        continue

    print(f"")
    print(f"{'='*70}")
    print(f"[{file_idx}/{total_files}] 📄 Processing: {file_info['filename']}")
    print(f"  Type:   {file_info['type']}")
    print(f"  Size:   {file_info['size_mb']} MB")
    print(f"{'='*70}")

    try:
        # 1. Prepare file (split if needed)
        print(f"\n📦 Preparing file...")
        files_to_ocr = prepare_file_for_ocr(file_path, str(TEMP_DIR))

        if len(files_to_ocr) > 1:
            print(f"✓ PDF split: {len(files_to_ocr)} chunks")
        else:
            print(f"✓ File ready (no splitting needed)")

        # 2. Process all chunks
        all_results = []

        for chunk_idx, chunk_path in enumerate(files_to_ocr, 1):
            chunk_name = Path(chunk_path).name

            if len(files_to_ocr) > 1:
                print(f"\n  [{chunk_idx}/{len(files_to_ocr)}] 🔄 Processing chunk: {chunk_name}")
            else:
                print(f"\n🔄 Starting OCR...")

            # OCR processing
            result = process_single_pdf(
                pdf_id=f"{file_id}_chunk{chunk_idx}" if len(files_to_ocr) > 1 else file_id,
                pdf_path=chunk_path,
                api_key=MISTRAL_API_KEY,
                output_dir=str(DATA_OUTPUT),
                db_path=DB_PATH,
                model=MISTRAL_MODEL,
                delay_seconds=DELAY_SECONDS
            )

            all_results.append(result)
            total_api_calls += 1

            if result['success']:
                print(f"  ✓ OCR successful: {result['total_pages']} pages, {result['confidence']:.2f} confidence")
                if result['warnings']:
                    print(f"  ⚠️  Warnings: {len(result['warnings'])}")
            else:
                print(f"  ✗ Error: {result['error']}")
                total_errors += 1

        # 3. Summary for this file
        successful_chunks = sum(1 for r in all_results if r['success'])
        total_pages = sum(r.get('total_pages', 0) for r in all_results if r['success'])

        print(f"\n{'='*70}")
        if successful_chunks == len(all_results):
            print(f"✅ FILE COMPLETE: {file_info['filename']}")
            print(f"   Chunks: {len(all_results)}, Pages: {total_pages}")
            total_processed += 1
        else:
            print(f"⚠️  PARTIAL ERROR: {successful_chunks}/{len(all_results)} chunks successful")
            total_errors += 1
        print(f"{'='*70}")

    except Exception as e:
        print(f"\n{'='*70}")
        print(f"✗ ERROR processing {file_info['filename']}: {e}")
        print(f"{'='*70}")
        total_errors += 1

# Final statistics
end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\n\n")
print("="*70)
print("BATCH PROCESSING COMPLETE")
print("="*70)
print(f"📊 Statistics:")
print(f"  Total files:        {total_files}")
print(f"  Successful:         {total_processed}")
print(f"  Errors:             {total_errors}")
print(f"  Already processed:  {len(processed_files)}")
print(f"  API calls:          {total_api_calls}")
print(f"")
print(f"⏱️  Processing time: {duration:.1f}s ({duration/60:.1f} min)")
print(f"")

# Detailed statistics from database
db_stats = get_processing_stats(DB_PATH)
print(f"📈 Database statistics:")
print(f"  Total processed:    {db_stats['completed']}")
print(f"  Total errors:       {db_stats['error']}")
print(f"  Avg. time:          {db_stats['avg_duration_sec']:.1f}s")
print(f"  Avg. confidence:    {db_stats['avg_confidence']:.2f}")
print(f"  Total pages:        {db_stats['total_pages']}")
print(f"  Estimated cost:     ${db_stats['estimated_cost_usd']:.4f}")
print("="*70)
print(f"✓ Results saved in: {DATA_OUTPUT}")
print("="*70)

## Cell 4: Cleanup & Storage Management

Deletes temporary PDF chunk files from `data/tracking/pdf_chunks/` to free up disk space. Analyzes storage before/after and shows freed space.

**Preserved:** SQLite database (checkpoint system) | OCR results (`.md`, `.txt`, `.json`) | Original PDFs in `data/input/`

**Deleted:** Only temporary PDF chunks from the splitting process (Cell 3)

**This cell can be run independently** (only requires Cell 1 for imports) | **Dry run:** Set `dry_run=True` in `cleanup_temp_files()` for test without deletion

In [ ]:
# Minimal imports if Cell 1 not executed
if 'PROJECT_ROOT' not in globals():
    from pathlib import Path
    from utils import get_storage_stats, cleanup_temp_files
    PROJECT_ROOT = Path("..").resolve()
    DATA_TRACKING = PROJECT_ROOT / "data" / "tracking"

# Define temp directory
TEMP_DIR = DATA_TRACKING / "pdf_chunks"

# Storage statistics BEFORE cleanup
print("="*70)
print("STORAGE ANALYSIS")
print("="*70)

try:
    storage_before = get_storage_stats(str(PROJECT_ROOT))
    
    print(f"📊 Current sizes:")
    print(f"")
    print(f"  Input PDFs:       {storage_before['input']:>8.2f} MB")
    print(f"  Output files:     {storage_before['output']:>8.2f} MB")
    print(f"  Tracking DB:      {storage_before['database']:>8.2f} MB")
    print(f"  PDF chunks:       {storage_before['chunks']:>8.2f} MB  ⚠️")
    print(f"  {'-'*40}")
    print(f"  TOTAL:            {storage_before['total']:>8.2f} MB")
    print(f"")
    
except Exception as e:
    logger.error(f"Error in storage analysis: {e}")
    print(f"⚠️  Warning: Storage statistics could not be calculated")
    print(f"   Error: {e}")
    print(f"")
    print(f"   Cleanup will still be attempted...")
    print(f"")
    storage_before = None

# Perform cleanup (even if stats failed)
try:
    if storage_before is None or storage_before.get('chunks', 0) >= 1.0:
        if storage_before and storage_before['chunks'] >= 1.0:
            print(f"⚠️  PDF chunks occupy {storage_before['chunks']:.2f} MB storage space.")
        else:
            print(f"🔄 Starting cleanup (storage info not available)...")
        
        print(f"")
        print("="*70)
        print("START CLEANUP")
        print("="*70)
        print(f"")

        # Perform cleanup
        try:
            cleanup_result = cleanup_temp_files(
                temp_dir=str(TEMP_DIR),
                dry_run=False  # Set to True for test run
            )

            print(f"")
            print("="*70)
            print("CLEANUP COMPLETE")
            print("="*70)
            print(f"")
            print(f"📋 Statistics:")
            print(f"  Found chunks:       {cleanup_result['files_found']}")
            print(f"  Deleted chunks:     {cleanup_result['files_deleted']}")
            print(f"  Freed:              {cleanup_result['space_freed_mb']:.2f} MB")
            print(f"")

            if cleanup_result['deleted_files']:
                print(f"🗑️  Deleted files:")
                for filename in cleanup_result['deleted_files'][:10]:  # Show max 10
                    print(f"  - {filename}")
                if len(cleanup_result['deleted_files']) > 10:
                    print(f"  ... and {len(cleanup_result['deleted_files']) - 10} more")
                print(f"")

        except Exception as e:
            logger.error(f"Cleanup failed: {e}")
            print(f"")
            print("="*70)
            print("✗ CLEANUP FAILED")
            print("="*70)
            print(f"")
            print(f"Error: {e}")
            print(f"")
            print(f"Possible causes:")
            print(f"  - File access issues (permissions)")
            print(f"  - Cloud sync active")
            print(f"  - Files used by another process")
            print(f"")
            print(f"💡 Tip: Try again later or delete chunks manually:")
            print(f"   {TEMP_DIR}")
            print("="*70)

        # Storage statistics AFTER cleanup (best effort)
        if storage_before is not None:
            try:
                storage_after = get_storage_stats(str(PROJECT_ROOT))

                print("="*70)
                print("STORAGE AFTER CLEANUP")
                print("="*70)
                print(f"")
                print(f"📊 New sizes:")
                print(f"")
                print(f"  Input PDFs:       {storage_after['input']:>8.2f} MB")
                print(f"  Output files:     {storage_after['output']:>8.2f} MB")
                print(f"  Tracking DB:      {storage_after['database']:>8.2f} MB")
                print(f"  PDF chunks:       {storage_after['chunks']:>8.2f} MB  ✓")
                print(f"  {'-'*40}")
                print(f"  TOTAL:            {storage_after['total']:>8.2f} MB")
                print(f"")
                print(f"💾 Saved: {storage_before['total'] - storage_after['total']:.2f} MB")
                print("="*70)
                print(f"✓ Cleanup successful!")
                print("="*70)
            except Exception as e:
                logger.warning(f"After-statistics could not be calculated: {e}")
                print(f"")
                print(f"⚠️  After-statistics not available (cleanup completed though)")
                print("="*70)
    else:
        print("✓ No PDF chunks found. Cleanup not needed.")
        print("="*70)

except Exception as e:
    # Unexpected error in entire cleanup workflow
    logger.error(f"Unexpected error in cleanup workflow: {e}")
    print(f"")
    print("="*70)
    print("✗ ERROR IN CLEANUP WORKFLOW")
    print("="*70)
    print(f"")
    print(f"An unexpected error occurred: {e}")
    print(f"")
    print(f"Cleanup could not be performed.")
    print(f"The pipeline will continue to work normally.")
    print("="*70)